# Notebook 3 — MLP on Tiny Shakespeare

The model stays the MLP from notebook 2; **only the data changes, to `tiny Shakespeare`.**

The task is still **fixed context -> next char**.

In [ ]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("shakespeare.txt").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "shakespeare.txt")
text = open("shakespeare.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab_size:", vocab_size)

text length: 1115394
vocab_size: 65


In [ ]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

## 1. Sliding-window dataset

In [ ]:
class CharSequenceNextCharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + self.block_size]
        return x, y

block_size = 16
dataset = CharSequenceNextCharDataset(data, block_size)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)
print("decoded x:", ''.join(itos[i.item()] for i in xb[0]))
print("decoded y:", itos[yb[0].item()])

xb.shape: torch.Size([128, 16])
yb.shape: torch.Size([128])
decoded x: he towns, as the
decoded y: y


## 2. MLP model

In [ ]:
class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([128, 65])
initial loss: 4.237505912780762


## 3. Training

In [ ]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(10):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.5999
epoch  1 | train loss 2.2543
epoch  2 | train loss 2.1378
epoch  3 | train loss 2.0756
epoch  4 | train loss 2.0134
epoch  5 | train loss 1.9674
epoch  6 | train loss 1.9491
epoch  7 | train loss 1.9025
epoch  8 | train loss 1.8838
epoch  9 | train loss 1.8584


## 4. Sampling

In [ ]:
@torch.no_grad()
def sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = [0] * block_size
    for ch in start_text:
        if ch in stoi:
            context = context[1:] + [stoi[ch]]
    out = list(start_text)
    for _ in range(max_new_tokens):
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1).item()
        out.append(itos[ix])
        context = context[1:] + [ix]
    return "".join(out)

print(sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

ROMEO:
Terl I reen oor rie that.

POMONES:
I, kerled Rind have quouchipld choecringer, to sir.

KING ROWHOMY:
My to renver ax is me:
Thass we shot you are nae, the bant, O' tey I'll foie, the cain,
What not must proons;
Cofeent ame po, that hears knag!
Ene you beriol marego nof him out of'l, for withs will yo s'ck dimess of ilus.

Figh of thou brower be wand gracus to me ioncom
And sobe a suad to wesure


## 5. Summary

- The same MLP model applies to a much larger text.
- What changes is the dataset.
- But a fixed-context MLP handles long context poorly.